In [1]:
import pandas as pd

In [2]:
#https://machinelearningmastery.com/time-series-forecasting-methods-in-python-cheat-sheet/

In [3]:
dataset=pd.read_csv("Tatacoffee13_21.csv")

In [4]:
dataset

,Date,Open,High,Low,Close
0,2013-01-01,1410.60,1427.90,1408.30,1415.10
1,2013-01-02,1421.00,1626.60,1416.15,1607.40
2,2013-01-03,1632.55,1673.90,1613.05,1626.20
3,2013-01-04,1627.75,1627.75,1574.60,1579.05
4,2013-01-07,1580.00,1639.50,1565.50,1595.65
...,...,...,...,...,...
2220,2021-12-22,202.90,207.80,201.35,205.00
2221,2021-12-23,206.00,206.85,202.05,202.95
2222,2021-12-24,203.90,203.90,199.35,201.00
2223,2021-12-27,200.00,222.00,196.00,218.35


In [5]:
dataset["Open"].value_counts()

Open
91.00     11
90.50     10
92.50      9
93.00      9
91.50      9
          ..
227.40     1
230.40     1
225.00     1
219.35     1
212.00     1
Name: count, Length: 1401, dtype: int64

In [6]:
dataset.isnull().sum()

Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64

In [7]:
timedata=pd.DataFrame()


In [8]:
timedata["Date"]=dataset["Date"]
timedata["Close"]=dataset["Close"]

In [9]:
data=timedata["Close"]

In [10]:
timedata

,Date,Close
0,2013-01-01,1415.10
1,2013-01-02,1607.40
2,2013-01-03,1626.20
3,2013-01-04,1579.05
4,2013-01-07,1595.65
...,...,...
2220,2021-12-22,205.00
2221,2021-12-23,202.95
2222,2021-12-24,201.00
2223,2021-12-27,218.35


In [11]:
data=timedata["Close"]

In [12]:
data_train=data.head(1500)

In [13]:
data_test=data.tail(509)

In [15]:
orders=[(1,0,2),(1,0,1),(2,0,1),(1,0,1)]

orderslist=[]
rscorelist=[]

for i in orders:

    orderslist.append(i)

    from statsmodels.tsa.arima.model import ARIMA
    model = ARIMA(data, order=i)
    model_fit = model.fit()

    y_pred = model_fit.predict(0, len(data)-1)

    from sklearn.metrics import r2_score
    r2 = r2_score(data, y_pred)

    rscorelist.append(r2)

    print("Order={} ,r2={}".format(i,r2))

Order=(1, 0, 2) ,r2=0.9944594472201003
Order=(1, 0, 1) ,r2=0.9944589988317689
Order=(2, 0, 1) ,r2=0.9944595582083863
Order=(1, 0, 1) ,r2=0.9944589988317689


In [17]:
trends=['n','t','c','ct']
orders=[(0,0,0),(0,0,1),(2,0,1),(1,1,1)]
orderslist=[]
trendslist=[]
rscorelist=[]

for td in trends:
    for i in orders:
        trendslist.append(td)
        orderslist.append(i)

        from statsmodels.tsa.statespace.sarimax import SARIMAX
        model = SARIMAX(data_train, order=i, seasonal_order=(0,0,0,0), trend=td)
        model_fit = model.fit()

        y_pred = model_fit.predict(start=len(data_train), end=len(data_train)+len(data_test)-1)

        from sklearn.metrics import r2_score
        r2 = r2_score(data_test, y_pred)

        rscorelist.append(r2)

        print("Trend={}, Order={}, r2={}".format(td,i,r2))

Trend=n, Order=(0, 0, 0), r2=-6.543778588771632


C:\Anaconda3\envs\aiml\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Trend=n, Order=(0, 0, 1), r2=-6.538820521592072
Trend=n, Order=(2, 0, 1), r2=-1.1827707904690494
Trend=n, Order=(1, 1, 1), r2=-0.5537344466396079
Trend=t, Order=(0, 0, 0), r2=-23.525551997062703


C:\Anaconda3\envs\aiml\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Trend=t, Order=(0, 0, 1), r2=-23.445120544849114
Trend=t, Order=(2, 0, 1), r2=-13.95512161281374
Trend=t, Order=(1, 1, 1), r2=-32.67560448652703
Trend=c, Order=(0, 0, 0), r2=-38.63184107116414
Trend=c, Order=(0, 0, 1), r2=-38.18091700092741
Trend=c, Order=(2, 0, 1), r2=-2.782029379001742
Trend=c, Order=(1, 1, 1), r2=-37.30708524866363
Trend=ct, Order=(0, 0, 0), r2=-153.40984293757083
Trend=ct, Order=(0, 0, 1), r2=-150.83162797011568
Trend=ct, Order=(2, 0, 1), r2=-98.0547519116158
Trend=ct, Order=(1, 1, 1), r2=-10.603888770718495


In [18]:


rscorelist

[-6.543778588771632,
 -6.538820521592072,
 -1.1827707904690494,
 -0.5537344466396079,
 -23.525551997062703,
 -23.445120544849114,
 -13.95512161281374,
 -32.67560448652703,
 -38.63184107116414,
 -38.18091700092741,
 -2.782029379001742,
 -37.30708524866363,
 -153.40984293757083,
 -150.83162797011568,
 -98.0547519116158,
 -10.603888770718495]

In [19]:
result=pd.DataFrame()

In [20]:
result.insert(0,"Trend",trendslist)
result.insert(1,"Order",orderslist)
result.insert(2,"R_score",rscorelist)

In [21]:
result

,Trend,Order,R_score
0,n,"(0, 0, 0)",-6.543779
1,n,"(0, 0, 1)",-6.538821
2,n,"(2, 0, 1)",-1.182771
3,n,"(1, 1, 1)",-0.553734
4,t,"(0, 0, 0)",-23.525552
5,t,"(0, 0, 1)",-23.445121
6,t,"(2, 0, 1)",-13.955122
7,t,"(1, 1, 1)",-32.675604
8,c,"(0, 0, 0)",-38.631841
9,c,"(0, 0, 1)",-38.180917


In [22]:
model_fit.predict(len(data), len(data)+5)

2225    884.993336
2226    886.747413
2227    888.503312
2228    890.261033
2229    892.020578
2230    893.781944
Name: predicted_mean, dtype: float64